# BioHub — sonde légère des données

Ce notebook inspecte les métadonnées et quelques frames sous-échantillonnées. Il ne charge jamais les 87+ Go en mémoire et n'entraîne aucun modèle.

Les images de la compétition sont stockées comme **Zarr v3** avec le tableau principal dans `0/` et un chunk Blosc2 par timepoint. Le lecteur ci-dessous ouvre directement `0/zarr.json` et les chunks `0/c/{t}/0/0/0`; il ne dépend donc pas du package `zarr`.


In [ ]:
from __future__ import annotations

from pathlib import Path
import importlib
import json
import subprocess
import sys

def ensure_blosc2() -> None:
    try:
        import blosc2  # noqa: F401
        return
    except ModuleNotFoundError:
        pass

    try:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "--quiet", "blosc2"]
        )
        importlib.invalidate_caches()
        import blosc2  # noqa: F401
    except Exception as exc:
        raise RuntimeError(
            "Le package blosc2 est absent. Active temporairement Internet dans "
            "Notebook options, relance cette cellule, ou attache un Dataset Kaggle "
            "contenant une wheel blosc2 compatible."
        ) from exc

ensure_blosc2()

import blosc2
print("blosc2 version:", getattr(blosc2, "__version__", "unknown"))


In [ ]:
from dataclasses import dataclass

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

COMPETITION_SLUG = "biohub-cell-tracking-during-development"
CANDIDATE_ROOTS = [
    Path("/kaggle/input/competitions") / COMPETITION_SLUG,
    Path("/kaggle/input") / COMPETITION_SLUG,
]

DATA_ROOT = next((p for p in CANDIDATE_ROOTS if p.exists()), None)
if DATA_ROOT is None:
    raise FileNotFoundError(
        "Competition mount not found. Attach the BioHub competition data to this notebook."
    )

TRAIN_ROOT = DATA_ROOT / "train"
TEST_ROOT = DATA_ROOT / "test"
WORK_ROOT = Path("/kaggle/working")
print("DATA_ROOT:", DATA_ROOT)


In [ ]:
def discover_stores(root: Path, suffix: str) -> list[Path]:
    if not root.exists():
        return []
    return sorted(path for path in root.iterdir() if path.name.endswith(suffix))

train_zarr = discover_stores(TRAIN_ROOT, ".zarr")
train_geff = discover_stores(TRAIN_ROOT, ".geff")
test_zarr = discover_stores(TEST_ROOT, ".zarr")

print(f"train zarr: {len(train_zarr)}")
print(f"train geff: {len(train_geff)}")
print(f"test zarr : {len(test_zarr)}")
print("train examples:", [p.name for p in train_zarr[:5]])
print("test examples :", [p.name for p in test_zarr[:5]])


In [ ]:
@dataclass(frozen=True)
class CompetitionImage:
    store_path: Path
    array_path: str
    shape: tuple[int, int, int, int]
    chunks: tuple[int, int, int, int] | None
    dtype: np.dtype

    @property
    def n_frames(self) -> int:
        return int(self.shape[0])

    def frame(self, t: int) -> np.ndarray:
        if not 0 <= t < self.n_frames:
            raise IndexError(f"timepoint {t} outside [0, {self.n_frames})")

        chunk_path = self.store_path / self.array_path / "c" / str(t) / "0" / "0" / "0"
        if not chunk_path.exists():
            raise FileNotFoundError(f"Chunk not found: {chunk_path}")

        raw = chunk_path.read_bytes()
        decoded = blosc2.decompress(raw)
        values = np.frombuffer(decoded, dtype=self.dtype)
        expected_shape = self.shape[1:]
        expected_size = int(np.prod(expected_shape))

        if values.size != expected_size:
            raise RuntimeError(
                f"Decoded chunk has {values.size} values; expected {expected_size} "
                f"for shape {expected_shape}."
            )
        return values.reshape(expected_shape).copy()


def open_competition_image(store_path: Path) -> CompetitionImage:
    # The competition stores the main Zarr v3 array below <dataset>.zarr/0.
    array_path = "0"
    metadata_path = store_path / array_path / "zarr.json"

    if not metadata_path.exists():
        available = [str(path.relative_to(store_path)) for path in store_path.rglob("zarr.json")]
        raise FileNotFoundError(
            f"Missing {metadata_path}. Zarr metadata found below the store: {available[:10]}"
        )

    metadata = json.loads(metadata_path.read_text())
    shape = tuple(int(value) for value in metadata["shape"])
    if len(shape) != 4:
        raise ValueError(f"Expected (T, Z, Y, X), got {shape} in {metadata_path}")

    dtype = np.dtype(metadata["data_type"])
    chunk_shape = (
        metadata.get("chunk_grid", {})
        .get("configuration", {})
        .get("chunk_shape")
    )
    chunks = tuple(int(value) for value in chunk_shape) if chunk_shape else None

    return CompetitionImage(
        store_path=store_path,
        array_path=array_path,
        shape=shape,
        chunks=chunks,
        dtype=dtype,
    )


rows = []
images: dict[tuple[str, str], CompetitionImage] = {}

for split, stores in (("train", train_zarr), ("test", test_zarr)):
    for store in stores:
        image = open_competition_image(store)
        images[(split, store.stem)] = image
        rows.append(
            {
                "split": split,
                "dataset": store.stem,
                "array_path": image.array_path,
                "shape": image.shape,
                "chunks": image.chunks,
                "dtype": str(image.dtype),
                "nbytes_logical_gb": float(
                    np.prod(image.shape) * image.dtype.itemsize / 1e9
                ),
            }
        )

inventory = pd.DataFrame(rows)
display(inventory)
inventory.to_csv(WORK_ROOT / "biohub_data_inventory.csv", index=False)


In [ ]:
if not train_zarr:
    raise RuntimeError("No training Zarr stores found.")

SAMPLE_STORE = train_zarr[0]
IMAGE = images[("train", SAMPLE_STORE.stem)]

print(
    "Sample:",
    SAMPLE_STORE.name,
    "array:",
    IMAGE.array_path,
    "shape:",
    IMAGE.shape,
    "chunks:",
    IMAGE.chunks,
)

T, Z, Y, X = IMAGE.shape
sampled_times = sorted({0, T // 2, T - 1})
stride_z = max(1, Z // 64)
stride_y = max(1, Y // 512)
stride_x = max(1, X // 512)

sample_rows = []
projections = {}

for t in sampled_times:
    full_frame = IMAGE.frame(t)
    volume = np.asarray(
        full_frame[::stride_z, ::stride_y, ::stride_x],
        dtype=np.float32,
    )
    del full_frame

    finite = volume[np.isfinite(volume)]
    quantiles = (
        np.quantile(finite, [0.5, 0.9, 0.99, 0.999])
        if finite.size
        else [np.nan] * 4
    )
    sample_rows.append(
        {
            "dataset": SAMPLE_STORE.stem,
            "t": t,
            "sample_shape": volume.shape,
            "q50": quantiles[0],
            "q90": quantiles[1],
            "q99": quantiles[2],
            "q999": quantiles[3],
            "mean": float(np.nanmean(volume)),
            "std": float(np.nanstd(volume)),
        }
    )
    projections[t] = np.nanmax(volume, axis=0)

sample_stats = pd.DataFrame(sample_rows)
display(sample_stats)
sample_stats.to_csv(WORK_ROOT / "biohub_sample_intensity_stats.csv", index=False)


In [ ]:
for t, projection in projections.items():
    plt.figure(figsize=(8, 8))
    lo, hi = np.nanquantile(projection, [0.01, 0.999])
    plt.imshow(projection, vmin=lo, vmax=hi)
    plt.title(f"{SAMPLE_STORE.stem} — t={t} — projection max Z sous-échantillonnée")
    plt.axis("off")
    plt.show()


In [ ]:
report = {
    "data_root": str(DATA_ROOT),
    "storage_format": "Zarr v3 / direct Blosc2 chunk reader",
    "train_zarr_count": len(train_zarr),
    "train_geff_count": len(train_geff),
    "test_zarr_count": len(test_zarr),
    "sample_dataset": SAMPLE_STORE.stem,
    "sample_array_path": IMAGE.array_path,
    "sample_shape": list(map(int, IMAGE.shape)),
    "sample_chunks": list(map(int, IMAGE.chunks)) if IMAGE.chunks else None,
    "sampled_times": sampled_times,
    "strides_zyx": [stride_z, stride_y, stride_x],
}
(WORK_ROOT / "biohub_data_probe.json").write_text(json.dumps(report, indent=2))
print(json.dumps(report, indent=2))
print("Artifacts written to /kaggle/working")
